# Merging FRED and SBA Data


In [33]:
import pandas as pd

### Exploring structure of FRED dataset

In [34]:
fred_data = pd.read_csv('../data/raw/fred_lookup.csv')

In [35]:
fred_data.head()

,state,year,state_unemployment,state_median_income,state_per_capita_income,state_gdp,state_nonfarm_employment,national_unemployment,fed_funds_rate,cpi,mortgage_30yr,consumer_confidence,treasury_10yr,unemployment_vs_national
0,AL,2010,10.3750,56840.0,33848.0,177510.7,1889.7667,9.6083,0.1750,218.0762,4.6898,71.8417,3.2151,0.7667
1,AL,2011,9.5583,57360.0,34884.0,182896.7,1890.1000,8.9333,0.1017,224.9230,4.4479,67.3500,2.7816,0.6250
2,AL,2012,8.1750,57430.0,35559.0,188280.7,1905.8000,8.0750,0.1400,229.5861,3.6575,76.5417,1.8034,0.1000
3,AL,2013,7.3583,61770.0,35706.0,193981.5,1924.1250,7.3583,0.1075,232.9517,3.9756,79.2083,2.3502,0.0000
4,AL,2014,6.7667,54420.0,36722.0,197064.4,1943.9083,6.1583,0.0892,236.7150,4.1689,84.1250,2.5396,0.6084


In [36]:
fred_data.shape

(510, 14)

In [37]:
fred_data.columns

Index(['state', 'year', 'state_unemployment', 'state_median_income',
       'state_per_capita_income', 'state_gdp', 'state_nonfarm_employment',
       'national_unemployment', 'fed_funds_rate', 'cpi', 'mortgage_30yr',
       'consumer_confidence', 'treasury_10yr', 'unemployment_vs_national'],
      dtype='object')

In [38]:
fred_data.describe()

,year,state_unemployment,state_median_income,state_per_capita_income,state_gdp,state_nonfarm_employment,national_unemployment,fed_funds_rate,cpi,mortgage_30yr,consumer_confidence,treasury_10yr,unemployment_vs_national
count,510.000000,510.000000,510.000000,510.000000,5.100000e+02,510.000000,510.000000,510.000000,510.000000,510.000000,510.000000,510.00000,510.000000
mean,2014.500000,5.783351,73829.039216,46756.368627,3.513266e+05,2758.050670,6.220820,0.613260,237.113230,4.091450,85.496680,2.40477,-0.437469
std,2.875101,2.247102,12766.999087,8776.804599,4.417518e+05,2984.082030,2.042815,0.742994,10.951355,0.344153,10.667315,0.43699,1.428223
min,2010.000000,2.100000,42210.000000,31213.000000,2.747830e+04,283.583300,3.675000,0.089200,218.076200,3.654000,67.350000,1.80340,-5.950000
25%,2012.000000,4.016700,64207.500000,40310.000000,8.715110e+04,726.049975,4.358300,0.107500,229.586100,3.850600,76.541700,2.13830,-1.208300
50%,2014.500000,5.350000,72890.000000,45466.000000,2.067514e+05,1876.175000,5.716650,0.157500,236.858400,3.982700,87.983350,2.33985,-0.270800
75%,2017.000000,7.222925,82762.500000,51672.500000,4.290653e+05,3392.043775,8.075000,1.001700,245.121000,4.447900,95.983300,2.78160,0.464625
max,2019.000000,13.250000,115800.000000,83675.000000,3.068630e+06,17432.116700,9.608300,2.158300,255.652600,4.689800,98.366700,3.21510,4.050000


In [39]:
# finding missing fred data
missing_fred_data = fred_data.isnull().sum()
print(missing_fred_data)

state                       0
year                        0
state_unemployment          0
state_median_income         0
state_per_capita_income     0
state_gdp                   0
state_nonfarm_employment    0
national_unemployment       0
fed_funds_rate              0
cpi                         0
mortgage_30yr               0
consumer_confidence         0
treasury_10yr               0
unemployment_vs_national    0
dtype: int64


### Merging FRED and SBA datasets

In [40]:
sba_data = pd.read_csv('../data/clean/sba_data_cleaned.csv', low_memory=False)

In [41]:
sba_data.isnull().sum().sum()

np.int64(432)

In [42]:
# Joining on state and year matches
combined = sba_data.merge(
    fred_data,
    left_on=["borrstate", "approvalfiscalyear"],
    right_on=["state", "year"],
    how="left"
)

# Drop the duplicate join columns from FRED side
combined = combined.drop(columns=["state", "year"])

# Verify the join results
print(f"Rows before join: {len(sba_data)}")
print(f"Rows after join: {len(combined)}")
print(f"New nulls: {combined.isnull().sum().sum()}")
print(f"New columns: {combined.shape[1]}")

Rows before join: 422423
Rows after join: 422423
New nulls: 47400
New columns: 28


In [43]:
# Finding mismatches causing nulls
sba_states = set(sba_data["borrstate"].unique())
fred_states = set(fred_data["state"].unique())

print("States in SBA but NOT in FRED:")
print(sba_states - fred_states)

print("\nStates in FRED but NOT in SBA:")
print(fred_states - sba_states)

# How many loans are in the unmatched states?
unmatched = sba_data[~sba_data["borrstate"].isin(fred_states)]
print(f"\nLoans in unmatched states: {len(unmatched)}")
print(f"\nBreakdown:")
print(unmatched["borrstate"].value_counts())

States in SBA but NOT in FRED:
{'MP', 'VI', 'MH', 'FM', 'GU', 'PR'}

States in FRED but NOT in SBA:
set()

Loans in unmatched states: 3914

Breakdown:
borrstate
PR    3455
GU     374
VI      76
MP       5
FM       3
MH       1
Name: count, dtype: int64


Places causing missing issue: Puerto Rico (PR), Guam (GU), U.S. Virgin Islands (VI), Northern Mariana Islands (MP), Federated States of Micronesia (FM), and Marshall Islands (MH)
- Need to drop them

In [44]:
# Dropping rows with unmatched states (those with nulls in state_unemployment)
combined = combined.dropna(subset=["state_unemployment"])

print(f"Rows after dropping unmatched: {len(combined)}")
print(f"Remaining nulls: {combined.isnull().sum().sum()}")

Rows after dropping unmatched: 418509
Remaining nulls: 425


In [45]:
print(f"Final shape: {combined.shape}")
print(f"Final nulls:\n{combined.isnull().sum()}")
print(f"Final null total: {combined.isnull().sum().sum()}")

Final shape: (418509, 28)
Final nulls:
borrstate                     0
grossapproval                 0
sbaguaranteedapproval         0
approvalfiscalyear            0
initialinterestrate           0
terminmonths                  0
naicsdescription            425
businesstype                  0
revolverstatus                0
jobssupported                 0
guarantee_pct                 0
naics_sector                  0
projectinstate                0
bankinstate                   0
default                       0
variable_rate                 0
state_unemployment            0
state_median_income           0
state_per_capita_income       0
state_gdp                     0
state_nonfarm_employment      0
national_unemployment         0
fed_funds_rate                0
cpi                           0
mortgage_30yr                 0
consumer_confidence           0
treasury_10yr                 0
unemployment_vs_national      0
dtype: int64
Final null total: 425


No new nulls presents, good to proceed

### Saving data

In [47]:
# Save the combined dataset
combined.to_csv('../data/clean/sba_fred_combined.csv', index=False)